In [32]:
import sys
print(sys.executable)
from imutils import paths,face_utils,resize
import numpy as np
import imutils
import cv2
import os
import dlib
from dlib import shape_predictor
#!pip install tqdm

from tqdm import tqdm

/Users/pangwenjing/opt/anaconda3/envs/newenv/bin/python


In [33]:
def face_detection(image):
  cascadePath = "haarcascade_frontalface_default.xml"
  detector = cv2.CascadeClassifier(cascadePath)

  gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  rects = detector.detectMultiScale(gray, scaleFactor=1.05,
	minNeighbors=10, minSize=(30, 30),
	flags=cv2.CASCADE_SCALE_IMAGE)

  return rects

In [46]:
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("/Users/pangwenjing/Desktop/capstone/capstone/dlib-models-master/shape_predictor_68_face_landmarks.dat")
  # You need to download this pre-trained model


def align_face(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    rects = detector(gray, 1)

    for rect in rects:
        shape = predictor(gray, rect)
        shape = face_utils.shape_to_np(shape)

        print("# Extract the left and right eye (x, y)-coordinates")
        (lStart, lEnd) = face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]
        (rStart, rEnd) = face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]
        leftEyePts = shape[lStart:lEnd]
        rightEyePts = shape[rStart:rEnd]
        print("left-eye:",lStart)
        print("righ-eye:",lEnd)

        print("# Compute the center of mass for each eye")
        leftEyeCenter = leftEyePts.mean(axis=0).astype("int")
        rightEyeCenter = rightEyePts.mean(axis=0).astype("int")
        print("leftEyeCenter:",leftEyeCenter)
        print("rightEyeCenter:",rightEyeCenter)

        print("# Compute the angle between the eye centroids")
        dY = rightEyeCenter[1] - leftEyeCenter[1]
        dX = rightEyeCenter[0] - leftEyeCenter[0]
        angle = np.degrees(np.arctan2(dY, dX))
        print(angle)

        print("# Compute the desired right eye x-coordinate based on the desired x-coordinate")
        desiredRightEyeX = 1.0 - 0.35
        print(desiredRightEyeX)

        # Determine the scale of the new resulting image by taking the ratio of the distance between the eyes
        # in the current image to the ratio of distance in the desired image
        dist = np.sqrt((dX ** 2) + (dY ** 2))
        desiredDist = (desiredRightEyeX - 0.35)
        desiredDist *= 48  # Width of the new image
        scale = desiredDist / dist

        # Compute center (x, y)-coordinates (i.e., the median point) between the two eyes in the input image
        #eyesCenter = ((leftEyeCenter[0] + rightEyeCenter[0]) // 2, (leftEyeCenter[1] + rightEyeCenter[1]) // 2)
        
        eyesCenter = np.mean([leftEyeCenter, rightEyeCenter], axis=0).astype("int")
        eyesCenter = (int(eyesCenter[0]), int(eyesCenter[1]))
        # print("eyecenter",eyesCenter[:10])
        if angle < -45:
          angle = 90 + angle
        elif angle > 45:
          angle = angle - 90

        # print(f"Type of eyesCenter: {type(eyesCenter)}, Value: {eyesCenter}")
        # print(f"Type of angle: {type(angle)}, Value: {angle}")
        # print(f"Type of scale: {type(scale)}, Value: {scale}")

        M = cv2.getRotationMatrix2D(eyesCenter, angle, scale)

    


        # Grab the rotation matrix for rotating and scaling the face
        M = cv2.getRotationMatrix2D(eyesCenter, angle, scale)

        # Update the translation component of the matrix
        tX = 48 * 0.5
        tY = 48 * 0.35
        M[0, 2] += (tX - eyesCenter[0])
        M[1, 2] += (tY - eyesCenter[1])

        # Apply the affine transformation
        (w, h) = (48, 48)
        output = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC)

        return output


In [43]:
from imutils import paths
import os
import cv2
from tqdm import tqdm
import numpy as np

def load_face_dataset(inputPath, minSamples=15):
    print("Loading images...")
    imagePaths = list(paths.list_images(inputPath))
    names = [p.split(os.path.sep)[-2] for p in imagePaths]
    (names, counts) = np.unique(names, return_counts=True)
    names = names.tolist()

    faces = []
    labels = []
    label_counts = {}

    # Loop over all of the image paths
    for imagePath in tqdm(imagePaths, desc="Processing images"):
        # Read the image and grab the image label
        image = cv2.imread(imagePath)
        name = imagePath.split(os.path.sep)[-2]

        # Initialize count for new label
        if name not in label_counts:
            label_counts[name] = 0

        # Check if already processed enough images for this label
        if label_counts[name] >= minSamples:
            continue

        # Update count for this label
        label_counts[name] += 1

        # Preprocess the face and add it to the list
        alignedFace = align_face(image)
        if alignedFace is not None:
            faceROI = cv2.cvtColor(alignedFace, cv2.COLOR_BGR2GRAY)
            faces.append(faceROI)
            labels.append(name)

    print("Converting to arrays...")
    faces = np.array(faces, dtype="object")
    labels = np.array(labels)

    print("Completed. Number of faces:", len(faces))
    return faces, labels


In [44]:
from skimage.feature import local_binary_pattern
import numpy as np

def extract_lbp_features(image, radius=1, points=5, method='uniform'):
    lbp = local_binary_pattern(image, points, radius, method)
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, points + 3), range=(0, points + 2))
    # Normalize the histogram
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)
    return hist


In [47]:
(faces , labels) = load_face_dataset("/Users/pangwenjing/Desktop/capstone/capstone/LBP_Baseline/Computer-Vision-Project/datasets/FER2013", minSamples=15)
print(set(labels))
lbp_features = []

for face in faces:
    hist = extract_lbp_features(face)
    lbp_features.append(hist)

# Convert the list to a NumPy array
lbp_features = np.array(lbp_features)


Loading images...


Processing images:   0%|          | 0/34490 [00:00<?, ?it/s]

# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [34 21]
rightEyeCenter: [14 18]
# Compute the angle between the eye centroids
-171.46923439005187
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [31 17]
rightEyeCenter: [11 17]
# Compute the angle between the eye centroids
180.0
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [34 15]
rightEyeCenter: [13 18]
# Compute the angle between the eye centroids
171.86989764584402
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42

Processing images:   1%|          | 272/34490 [00:00<00:23, 1434.34it/s]

# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [32 17]
rightEyeCenter: [14 16]
# Compute the angle between the eye centroids
-176.82016988013575
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65


...

# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [33 22]
rightEyeCenter: [15 18]
# Compute the angle between the eye centroids
-167.47119229084848
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [31 19]
rightEyeCenter: [14 19]
# Compute the angle between the eye centroids
180.0
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [31 16]
rightEyeCenter: [17 21]
# Compute the angle between the eye centroids
160.3461759419467
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42


Processing images:  21%|██        | 7273/34490 [00:03<00:12, 2109.82it/s]

# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [31 17]
rightEyeCenter: [14 18]
# Compute the angle between the eye centroids
176.63353933657018
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [32 20]
rightEyeCenter: [14 18]
# Compute the angle between the eye centroids
-173.6598082540901
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42
righ-eye: 48
# Compute the center of mass for each eye
leftEyeCenter: [32 16]
rightEyeCenter: [13 16]
# Compute the angle between the eye centroids
180.0
# Compute the desired right eye x-coordinate based on the desired x-coordinate
0.65
# Extract the left and right eye (x, y)-coordinates
left-eye: 42


Processing images: 100%|██████████| 34490/34490 [00:08<00:00, 4193.73it/s] 

Converting to arrays...
Completed. Number of faces: 71
{'neutral', 'angry', 'disgust', 'happy', 'surprise', 'fear', 'sad'}


In [38]:
%pip install pandas
import pandas as pd

# Assuming 'labels' is a list or a NumPy array containing your class labels
labels_series = pd.Series(labels)
class_distribution = labels_series.value_counts()
print(len(labels))
print(len(faces))
print(class_distribution)

Note: you may need to restart the kernel to use updated packages.
71
71
neutral     12
angry       12
surprise    11
fear        10
disgust     10
happy        9
sad          7
dtype: int64


In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(faces, labels, test_size=0.2, random_state=42)


In [40]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV


In [41]:
from sklearn.metrics import classification_report

# Assuming X_train is already loaded and shaped (number of samples, height, width, channels)
# Flatten the images for the SVM
X_train_flat = X_train.reshape((X_train.shape[0], -1))

# Initialize the classifier with probability calculation enabled
clf = SVC(kernel='linear', probability=True)

# Train the classifier on the flattened training data
clf.fit(X_train_flat, y_train)

# Predict on the flattened test set
# Ensure you have applied the same preprocessing to X_test as X_train
X_test_flat = X_test.reshape((X_test.shape[0], -1))
y_pred = clf.predict(X_test_flat)

# Evaluate the classifier's performance on the test set
print(classification_report(y_test, y_pred))



              precision    recall  f1-score   support

       angry       0.50      0.67      0.57         3
     disgust       0.00      0.00      0.00         1
        fear       0.00      0.00      0.00         2
       happy       0.50      0.33      0.40         3
     neutral       0.00      0.00      0.00         1
         sad       0.00      0.00      0.00         3
    surprise       0.00      0.00      0.00         2

    accuracy                           0.20        15
   macro avg       0.14      0.14      0.14        15
weighted avg       0.20      0.20      0.19        15



/Users/pangwenjing/opt/anaconda3/envs/newenv/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/pangwenjing/opt/anaconda3/envs/newenv/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/pangwenjing/opt/anaconda3/envs/newenv/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,